# NetShield — Phase 19: Domain-Adversarial Neural Network (DANN)

**Goal:** make the cross-dataset NIDS model genuinely more transferable from UNSW-NB15 to CICIDS2017.

This is a **new adaptation methodology**, not another round of adding features/files. DANN learns a representation that simultaneously:
1. predicts the attack/benign label on the labeled UNSW source data, and
2. makes source and unlabeled CICIDS target representations difficult for a domain classifier to distinguish.

### Frozen evaluation protocol
- Source: official UNSW-NB15 training set, split 80/20 into source-fit/source-validation.
- Target: 600,000 CICIDS rows sampled **without using labels**; 300,000 adaptation rows + 300,000 frozen evaluation rows.
- Target labels are touched only after the model and source threshold are frozen.
- Primary benchmark: **CORAL + balanced**, previously F1 ≈ 0.3385 under the earlier protocol. This notebook also recomputes that benchmark on the exact same target split, so comparisons are apples-to-apples.
- Main representation: the verified 11-feature semantic representation.

**Important:** do not tune DANN hyperparameters using CICIDS evaluation labels. The target evaluation table is only for final reporting.

In [ ]:
import os, gc, json, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
except ImportError as e:
    raise ImportError(
        "PyTorch is required for Phase 19 (DANN). Install a compatible PyTorch build "
        "for your Python environment, then restart the kernel."
    ) from e

print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
# Project-root discovery
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

def find_first(candidates):
    for p in candidates:
        if p.exists():
            return p.resolve()
    return None

UNSW_ROOT = find_first([
    ROOT / "data" / "raw" / "UNSW-NB15",
    ROOT / "UNSW-NB15",
])
CICIDS_ROOT = find_first([
    ROOT / "data" / "raw" / "CICIDS2017",
    ROOT / "data" / "raw" / "CICIDS",
    ROOT / "CICIDS2017",
    ROOT / "CICIDS",
])

print("Working directory:", NOTEBOOK_DIR)
print("Project root:", ROOT)
print("UNSW root:", UNSW_ROOT)
print("CICIDS root:", CICIDS_ROOT)

if UNSW_ROOT is None:
    raise FileNotFoundError("UNSW-NB15 folder not found. Expected NetShield/data/raw/UNSW-NB15.")
if CICIDS_ROOT is None:
    raise FileNotFoundError("CICIDS2017 folder not found. Expected NetShield/data/raw/CICIDS2017 or NetShield/data/raw/CICIDS.")


## 1. Robust schema discovery

Earlier notebooks failed because normalized column names were passed to `read_csv(usecols=...)`, while pandas expects the **original CSV header strings**. Here we inspect the original header, map semantic names to the exact raw names, and pass those exact names to pandas.

In [ ]:
def norm(x):
    return str(x).replace("\ufeff", "").replace("\xa0", " ").strip().lower()

def actual_col(columns, wanted):
    w = norm(wanted)
    for c in columns:
        if norm(c) == w:
            return c
    return None

# Locate official UNSW train/test files by their actual names.
all_unsw = sorted(UNSW_ROOT.rglob("*.csv"))
print("UNSW CSVs:")
for p in all_unsw:
    print(" ", p.relative_to(UNSW_ROOT))

def has_label(p):
    try:
        cols = pd.read_csv(p, nrows=0).columns
        return actual_col(cols, "label") is not None
    except Exception:
        return False

labelled_unsw = [p for p in all_unsw if has_label(p)]
train_candidates = [p for p in labelled_unsw if "training" in norm(p.name) and "set" in norm(p.name)]
test_candidates = [p for p in labelled_unsw if "testing" in norm(p.name) and "set" in norm(p.name)]

if not train_candidates or not test_candidates:
    raise FileNotFoundError(
        "Could not identify UNSW training/testing CSVs. Inspect the printed file list and set the paths manually."
    )
UNSW_TRAIN_PATH = train_candidates[0]
UNSW_TEST_PATH = test_candidates[0]
print("UNSW train:", UNSW_TRAIN_PATH)
print("UNSW test:", UNSW_TEST_PATH)

# CICIDS files: only CSVs with a Label column.
cicids_files=[]
for p in sorted(CICIDS_ROOT.rglob("*.csv")):
    try:
        cols=pd.read_csv(p,nrows=0).columns
        if actual_col(cols,"Label") is not None:
            cicids_files.append(p)
    except Exception:
        pass

if not cicids_files:
    raise FileNotFoundError("No CICIDS CSV with a Label column was found.")
print("CICIDS files:")
for p in cicids_files:
    print(" ", p.name)


In [ ]:
# Load UNSW official train/test.
unsw_train = pd.read_csv(UNSW_TRAIN_PATH, low_memory=False)
unsw_test = pd.read_csv(UNSW_TEST_PATH, low_memory=False)

label_train = actual_col(unsw_train.columns, "label")
label_test = actual_col(unsw_test.columns, "label")
if label_train is None or label_test is None:
    raise KeyError("UNSW label column not found.")

unsw_train["label_binary"] = pd.to_numeric(unsw_train[label_train], errors="coerce").fillna(0).astype("int8")
unsw_test["label_binary"] = pd.to_numeric(unsw_test[label_test], errors="coerce").fillna(0).astype("int8")

print("UNSW train:", unsw_train.shape, unsw_train["label_binary"].value_counts().to_dict())
print("UNSW test:", unsw_test.shape, unsw_test["label_binary"].value_counts().to_dict())


## 2. Build the verified 11-feature semantic representation

These are the same semantic quantities already validated in the project. No new feature engineering is introduced here; the methodological change is the **domain-adversarial learning objective**.

In [ ]:
FEATURES = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate", "avg_packet_size"
]

def safe_div(a, b):
    a = pd.to_numeric(a, errors="coerce").to_numpy(dtype="float64")
    b = pd.to_numeric(b, errors="coerce").to_numpy(dtype="float64")
    out = np.full(a.shape, np.nan, dtype="float64")
    np.divide(a, b, out=out, where=np.isfinite(b) & (b != 0))
    return out

def build_unsw_common(df):
    cols={norm(c):c for c in df.columns}
    req=["dur","spkts","dpkts","sbytes","dbytes"]
    miss=[c for c in req if c not in cols]
    if miss: raise KeyError(f"UNSW common columns missing: {miss}")
    out=pd.DataFrame(index=df.index)
    out["flow_duration_sec"]=pd.to_numeric(df[cols["dur"]],errors="coerce")
    out["fwd_packets"]=pd.to_numeric(df[cols["spkts"]],errors="coerce")
    out["bwd_packets"]=pd.to_numeric(df[cols["dpkts"]],errors="coerce")
    out["fwd_bytes"]=pd.to_numeric(df[cols["sbytes"]],errors="coerce")
    out["bwd_bytes"]=pd.to_numeric(df[cols["dbytes"]],errors="coerce")
    out["total_packets"]=out["fwd_packets"]+out["bwd_packets"]
    out["total_bytes"]=out["fwd_bytes"]+out["bwd_bytes"]
    out["packet_rate"]=safe_div(out["total_packets"],out["flow_duration_sec"])
    out["fwd_packet_rate"]=safe_div(out["fwd_packets"],out["flow_duration_sec"])
    out["bwd_packet_rate"]=safe_div(out["bwd_packets"],out["flow_duration_sec"])
    out["avg_packet_size"]=safe_div(out["total_bytes"],out["total_packets"])
    return out[FEATURES]

def build_cicids_common(df):
    cols={norm(c):c for c in df.columns}
    req=["flow duration","total fwd packets","total backward packets","total length of fwd packets","total length of bwd packets","flow packets/s","fwd packets/s","bwd packets/s","average packet size"]
    miss=[c for c in req if c not in cols]
    if miss: raise KeyError(f"CICIDS common columns missing: {miss}")
    out=pd.DataFrame(index=df.index)
    out["flow_duration_sec"]=pd.to_numeric(df[cols["flow duration"]],errors="coerce")/1_000_000.0
    out["fwd_packets"]=pd.to_numeric(df[cols["total fwd packets"]],errors="coerce")
    out["bwd_packets"]=pd.to_numeric(df[cols["total backward packets"]],errors="coerce")
    out["fwd_bytes"]=pd.to_numeric(df[cols["total length of fwd packets"]],errors="coerce")
    out["bwd_bytes"]=pd.to_numeric(df[cols["total length of bwd packets"]],errors="coerce")
    out["total_packets"]=out["fwd_packets"]+out["bwd_packets"]
    out["total_bytes"]=out["fwd_bytes"]+out["bwd_bytes"]
    out["packet_rate"]=pd.to_numeric(df[cols["flow packets/s"]],errors="coerce")
    out["fwd_packet_rate"]=pd.to_numeric(df[cols["fwd packets/s"]],errors="coerce")
    out["bwd_packet_rate"]=pd.to_numeric(df[cols["bwd packets/s"]],errors="coerce")
    out["avg_packet_size"]=pd.to_numeric(df[cols["average packet size"]],errors="coerce")
    return out[FEATURES]

X_unsw_full=build_unsw_common(unsw_train)
y_unsw_full=unsw_train["label_binary"].to_numpy(dtype="int8")
print("UNSW semantic shape:",X_unsw_full.shape)


## 3. Create a fixed, label-blind CICIDS target pool

We sample 600,000 rows from the eight CICIDS CSVs proportionally to file size. **No CICIDS labels are used to construct the adaptation/evaluation split.** We keep the label only in a separate evaluation array.

In [ ]:
TARGET_POOL_N=600_000
CHUNK_SIZE=100_000

# First count rows without loading all feature columns.
file_rows={}
for path in cicids_files:
    hdr=pd.read_csv(path,nrows=0)
    lab=actual_col(hdr.columns,"Label")
    n=0
    for ch in pd.read_csv(path,usecols=[lab],chunksize=CHUNK_SIZE,low_memory=True):
        n += len(ch)
    file_rows[path]=n

total_rows=sum(file_rows.values())
pool_n=min(TARGET_POOL_N,total_rows)
quotas={}
remaining=pool_n
for i,path in enumerate(cicids_files):
    q=remaining if i==len(cicids_files)-1 else min(int(round(pool_n*file_rows[path]/total_rows)),remaining)
    quotas[path]=q
    remaining-=q

parts=[]
for path in cicids_files:
    hdr=pd.read_csv(path,nrows=0)
    cols=hdr.columns.tolist()
    lab=actual_col(cols,"Label")
    needed=[
        actual_col(cols,"Flow Duration"),
        actual_col(cols,"Total Fwd Packets"),
        actual_col(cols,"Total Backward Packets"),
        actual_col(cols,"Total Length of Fwd Packets"),
        actual_col(cols,"Total Length of Bwd Packets"),
        actual_col(cols,"Flow Packets/s"),
        actual_col(cols,"Fwd Packets/s"),
        actual_col(cols,"Bwd Packets/s"),
        actual_col(cols,"Average Packet Size"),
        lab
    ]
    if any(x is None for x in needed):
        raise KeyError(f"Missing semantic CICIDS column in {path.name}: {needed}")
    chunks=[]
    for ch in pd.read_csv(path,usecols=needed,chunksize=CHUNK_SIZE,low_memory=True):
        chunks.append(ch)
    df=pd.concat(chunks,ignore_index=True)
    n_take=min(quotas[path],len(df))
    df=df.sample(n=n_take,random_state=RANDOM_STATE)
    df["label_binary"]=df[lab].astype(str).str.strip().str.upper().ne("BENIGN").astype("int8")
    parts.append(df)
    print("sampled",n_take,path.name)
    del chunks,df
    gc.collect()

cicids_pool=pd.concat(parts,ignore_index=True)
cicids_pool=cicids_pool.sample(n=pool_n,random_state=RANDOM_STATE).reset_index(drop=True)

# Label-blind 50/50 split: labels are NOT used in train_test_split.
perm=np.random.default_rng(RANDOM_STATE).permutation(len(cicids_pool))
adapt_idx=perm[:min(300_000,len(perm)//2)]
eval_idx=perm[min(300_000,len(perm)//2):min(600_000,len(perm))]

X_cic_all=build_cicids_common(cicids_pool)
y_cic_all=cicids_pool["label_binary"].to_numpy(dtype="int8")
X_adapt=X_cic_all.iloc[adapt_idx].reset_index(drop=True)
X_eval=X_cic_all.iloc[eval_idx].reset_index(drop=True)
y_eval=y_cic_all[eval_idx].copy()

print("Target adaptation:",X_adapt.shape)
print("Target evaluation:",X_eval.shape)
print("Target eval attack rate (REPORTING ONLY):",round(float(y_eval.mean()),6))


## 4. Source fit/validation split and leakage-safe preprocessing

The scaler and imputer are fitted only on UNSW source-fit data. For DANN, the unlabeled target adaptation data are then transformed with the same source preprocessing. This avoids leaking target evaluation statistics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix)

fit_idx,val_idx=train_test_split(
    np.arange(len(X_unsw_full)),test_size=0.20,random_state=RANDOM_STATE,stratify=y_unsw_full
)

imp=SimpleImputer(strategy="median")
scaler=StandardScaler()
X_fit=imp.fit_transform(X_unsw_full.iloc[fit_idx]).astype("float32")
X_val=imp.transform(X_unsw_full.iloc[val_idx]).astype("float32")
X_adapt_z=scaler.fit_transform(X_fit).astype("float32")
X_val_z=scaler.transform(X_val).astype("float32")
X_adapt_z=scaler.transform(imp.transform(X_adapt)).astype("float32")
X_eval_z=scaler.transform(imp.transform(X_eval)).astype("float32")
y_fit=y_unsw_full[fit_idx].astype("int64")
y_val=y_unsw_full[val_idx].astype("int64")

print("Source fit:",X_fit.shape,"source validation:",X_val.shape)
print("Target adaptation:",X_adapt_z.shape,"target eval:",X_eval_z.shape)


## 5. DANN implementation

The gradient-reversal layer makes the feature extractor learn two competing objectives:
- **label discrimination:** useful features for detecting attacks,
- **domain confusion:** features that do not reveal whether a row came from UNSW or CICIDS.

The target domain labels used here are only 0/1 indicators of **source vs target dataset**, not attack labels.

In [ ]:
from torch.autograd import Function

class GradientReversal(Function):
    @staticmethod
    def forward(ctx,x,lambda_):
        ctx.lambda_=lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx,grad_output):
        return -ctx.lambda_*grad_output,None

def grad_reverse(x,lambda_):
    return GradientReversal.apply(x,lambda_)

class DANN(nn.Module):
    def __init__(self,n_features):
        super().__init__()
        self.feature=nn.Sequential(
            nn.Linear(n_features,64),nn.ReLU(),nn.Dropout(0.10),
            nn.Linear(64,32),nn.ReLU()
        )
        self.label_head=nn.Linear(32,1)
        self.domain_head=nn.Sequential(
            nn.Linear(32,32),nn.ReLU(),nn.Dropout(0.10),nn.Linear(32,1)
        )
    def forward(self,x,lambda_=0.0):
        z=self.feature(x)
        y_logit=self.label_head(z).squeeze(1)
        d_logit=self.domain_head(grad_reverse(z,lambda_)).squeeze(1)
        return y_logit,d_logit

print("DANN architecture ready.")


In [ ]:
# Training configuration — selected a priori, not tuned on CICIDS evaluation labels.
EPOCHS=12
BATCH_SIZE=1024
LR=1e-3
WEIGHT_DECAY=1e-4
DOMAIN_WEIGHT=0.50
RANDOM_STATE=42

def make_loader(X,y,batch=BATCH_SIZE,shuffle=True):
    ds=TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(y))
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=0,pin_memory=torch.cuda.is_available())

source_loader=make_loader(X_fit,y_fit)
target_domain_loader=DataLoader(
    TensorDataset(torch.tensor(X_adapt_z,dtype=torch.float32)),
    batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available()
)

# Source-only validation threshold: choose F1 on UNSW validation, exactly as the project's transfer protocol requires.
def source_threshold(model):
    model.eval()
    with torch.no_grad():
        x=torch.tensor(X_val_z,dtype=torch.float32,device=DEVICE)
        p=torch.sigmoid(model.label_head(model.feature(x)).squeeze(1)).cpu().numpy()
    ts=np.linspace(0.05,0.95,91)
    scores=[f1_score(y_val,(p>=t).astype(int),zero_division=0) for t in ts]
    j=int(np.argmax(scores))
    return float(ts[j]),p

def train_dann():
    torch.manual_seed(RANDOM_STATE)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_STATE)
    model=DANN(X_fit.shape[1]).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    # Balanced source classification loss: inverse-frequency weighting.
    pos=max(1,int(y_fit.sum())); neg=max(1,int(len(y_fit)-y_fit.sum()))
    pos_weight=torch.tensor([neg/pos],dtype=torch.float32,device=DEVICE)
    label_loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    domain_loss_fn=nn.BCEWithLogitsLoss()
    history=[]
    n_batches=max(len(source_loader),len(target_domain_loader))
    for epoch in range(EPOCHS):
        model.train()
        src_iter=iter(source_loader); tgt_iter=iter(target_domain_loader)
        total_label=total_domain=0.0
        for step in range(n_batches):
            try: sx,sy=next(src_iter)
            except StopIteration: src_iter=iter(source_loader); sx,sy=next(src_iter)
            try: tx_tuple=next(tgt_iter)
            except StopIteration: tgt_iter=iter(target_domain_loader); tx_tuple=next(tgt_iter)
            tx=tx_tuple[0]
            sx,sy=sx.to(DEVICE),sy.float().to(DEVICE)
            tx=tx.to(DEVICE)
            p=(epoch*n_batches+step)/(EPOCHS*n_batches)
            lambda_grl=DOMAIN_WEIGHT*(2/(1+np.exp(-10*p))-1)
            xdom=torch.cat([sx,tx],dim=0)
            ydom=torch.cat([torch.zeros(len(sx),device=DEVICE),torch.ones(len(tx),device=DEVICE)])
            ylogit,dlogit=model(torch.cat([sx,tx],dim=0),lambda_=lambda_grl)
            # Classification loss is computed only on source rows.
            l_label=label_loss_fn(ylogit[:len(sx)],sy)
            l_domain=domain_loss_fn(dlogit,ydom)
            loss=l_label+DOMAIN_WEIGHT*l_domain
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
            total_label+=float(l_label.detach().cpu()); total_domain+=float(l_domain.detach().cpu())
        history.append({"epoch":epoch+1,"label_loss":total_label/n_batches,"domain_loss":total_domain/n_batches,"lambda":lambda_grl})
        print(f"epoch {epoch+1:02d}/{EPOCHS} label={history[-1]['label_loss']:.4f} domain={history[-1]['domain_loss']:.4f} lambda={lambda_grl:.3f}")
    return model,pd.DataFrame(history)

start=time.perf_counter()
dann_model,dann_history=train_dann()
print("Training seconds:",round(time.perf_counter()-start,2))


## 6. Freeze the source-derived threshold

The threshold is selected **only on UNSW validation**. We never select a CICIDS threshold.

In [ ]:
dann_threshold,val_prob=source_threshold(dann_model)
print("Frozen DANN threshold:",dann_threshold)
print("Source validation F1:",round(f1_score(y_val,(val_prob>=dann_threshold).astype(int)),6))


## 7. Recompute the CORAL + balanced benchmark on the exact same target split

This is essential. We are no longer comparing numbers produced under different target samples.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve

def fit_coral(Xs,Xt):
    mu_s=Xs.mean(axis=0); mu_t=Xt.mean(axis=0)
    Cs=np.cov(Xs,rowvar=False)+1e-5*np.eye(Xs.shape[1])
    Ct=np.cov(Xt,rowvar=False)+1e-5*np.eye(Xt.shape[1])
    def invsqrt(C):
        w,V=np.linalg.eigh(C); return (V*(1/np.sqrt(np.maximum(w,1e-8))))@V.T
    def sqrt(C):
        w,V=np.linalg.eigh(C); return (V*np.sqrt(np.maximum(w,1e-8)))@V.T
    A=invsqrt(Cs)@sqrt(Ct)
    return mu_s,mu_t,A

def apply_coral_source(X,mu_s,mu_t,A):
    return (X-mu_s)@A+mu_t

mu_s,mu_t,A=fit_coral(X_adapt_z,X_adapt_z)  # placeholder overwritten below
# CORAL source domain uses UNSW source-fit; target covariance comes from unlabeled CICIDS adaptation.
mu_s=X_fit.mean(axis=0); mu_t=X_adapt_z.mean(axis=0)
Cs=np.cov(X_fit,rowvar=False)+1e-5*np.eye(X_fit.shape[1])
Ct=np.cov(X_adapt_z,rowvar=False)+1e-5*np.eye(X_adapt_z.shape[1])
def mat_sqrt(C):
    w,V=np.linalg.eigh(C); return (V*np.sqrt(np.maximum(w,1e-8)))@V.T
def mat_invsqrt(C):
    w,V=np.linalg.eigh(C); return (V*(1/np.sqrt(np.maximum(w,1e-8))))@V.T
A=mat_invsqrt(Cs)@mat_sqrt(Ct)
X_fit_c=(X_fit-mu_s)@A+mu_t
X_eval_c=X_eval_z

rf=RandomForestClassifier(n_estimators=250,random_state=RANDOM_STATE,n_jobs=-1,max_features="sqrt",min_samples_leaf=2,class_weight="balanced")
rf.fit(X_fit_c,y_fit)
# Source validation must also be represented in the aligned source coordinate system.
X_val_c=(X_val-mu_s)@A+mu_t
pv=rf.predict_proba(X_val_c)[:,1]
ts=np.linspace(.05,.95,91)
f1s=[f1_score(y_val,(pv>=t).astype(int),zero_division=0) for t in ts]
coral_threshold=float(ts[int(np.argmax(f1s))])

# For target, represent target rows in target standardized coordinates (no second A multiplication).
p_coral=rf.predict_proba(X_eval_z)[:,1]

def report(name,p,threshold):
    pred=(p>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_eval,pred,labels=[0,1]).ravel()
    return {
        "method":name,"threshold":threshold,"accuracy":accuracy_score(y_eval,pred),
        "precision":precision_score(y_eval,pred,zero_division=0),"recall":recall_score(y_eval,pred,zero_division=0),
        "f1":f1_score(y_eval,pred,zero_division=0),"macro_f1":f1_score(y_eval,pred,average="macro",zero_division=0),
        "pr_auc":average_precision_score(y_eval,p),"roc_auc":roc_auc_score(y_eval,p),
        "fpr":fp/(fp+tn) if (fp+tn) else 0.0,"tn":tn,"fp":fp,"fn":fn,"tp":tp,
        "predicted_attack_rate":float(pred.mean())
    }

bench=report("CORAL + balanced",p_coral,coral_threshold)
print(pd.DataFrame([bench]))


## 8. Evaluate DANN on the frozen CICIDS test partition

CICIDS labels enter the notebook here **for the first time in the DANN experiment**. They are not used for training, thresholding, or model selection.

In [ ]:
dann_model.eval()
with torch.no_grad():
    x=torch.tensor(X_eval_z,dtype=torch.float32,device=DEVICE)
    p_dann=torch.sigmoid(dann_model.label_head(dann_model.feature(x)).squeeze(1)).cpu().numpy()

dann_result=report("DANN",p_dann,dann_threshold)
coral_result=report("CORAL + balanced",p_coral,coral_threshold)
results=pd.DataFrame([coral_result,dann_result])
results["f1_gain_vs_CORAL_balanced"]=results["f1"]-float(coral_result["f1"])
results["pr_auc_gain_vs_CORAL_balanced"]=results["pr_auc"]-float(coral_result["pr_auc"])
results["roc_auc_gain_vs_CORAL_balanced"]=results["roc_auc"]-float(coral_result["roc_auc"])
display(results)


## 9. Domain-confusion diagnostic

A useful DANN sanity check is whether the learned features actually make source and target less distinguishable. We train a small post-hoc domain classifier on frozen representations. This is diagnostic only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Extract DANN features.
dann_model.eval()
with torch.no_grad():
    z_s=dann_model.feature(torch.tensor(X_fit,dtype=torch.float32,device=DEVICE)).cpu().numpy()
    z_t=dann_model.feature(torch.tensor(X_adapt_z,dtype=torch.float32,device=DEVICE)).cpu().numpy()

# Subsample equally for diagnostic speed.
rng=np.random.default_rng(RANDOM_STATE)
ns=min(100_000,len(z_s),len(z_t))
isrc=rng.choice(len(z_s),ns,replace=False); itgt=rng.choice(len(z_t),ns,replace=False)
Xd=np.vstack([z_s[isrc],z_t[itgt]]); yd=np.r_[np.zeros(ns,dtype=int),np.ones(ns,dtype=int)]
domain_probe=LogisticRegression(max_iter=300,random_state=RANDOM_STATE)
domain_probe.fit(Xd,yd)
probe_acc=accuracy_score(yd,domain_probe.predict(Xd))
print("Post-hoc source-vs-target domain probe accuracy:",round(probe_acc,4))
print("0.50 is ideal domain confusion; 1.00 means domains remain easily distinguishable.")


## 10. Save outputs and enforce the project decision rule

**Success criterion:** DANN must beat the exact-split CORAL + balanced benchmark on F1 while keeping the precision/FPR trade-off defensible. If it does not, we do not manufacture a gain by tuning on CICIDS labels.

In [ ]:
OUT=ROOT/"outputs"/"phase19_dann"
TABLES=OUT/"tables"
TABLES.mkdir(parents=True,exist_ok=True)

results.to_csv(TABLES/"01_phase19_dann_results.csv",index=False)
dann_history.to_csv(TABLES/"02_dann_training_history.csv",index=False)
config={
    "phase":19,"method":"DANN", "random_state":RANDOM_STATE,
    "target_pool_n":int(len(cicids_pool)),"target_adaptation_n":int(len(X_adapt_z)),"target_eval_n":int(len(X_eval_z)),
    "features":FEATURES,"epochs":EPOCHS,"batch_size":BATCH_SIZE,"learning_rate":LR,
    "weight_decay":WEIGHT_DECAY,"domain_weight":DOMAIN_WEIGHT,"device":str(DEVICE),
    "target_labels_used_for_training":False,"target_labels_used_for_threshold":False,
    "benchmark_f1":float(coral_result["f1"]),
    "dann_f1":float(dann_result["f1"]),
    "dann_gain":float(dann_result["f1"]-coral_result["f1"])
}
with open(TABLES/"03_phase19_config.json","w",encoding="utf-8") as f: json.dump(config,f,indent=2)

print("Saved outputs to:",TABLES)
print("\nDECISION")
if dann_result["f1"] > coral_result["f1"]:
    print("SUCCESS: DANN improved F1 over the exact-split CORAL + balanced benchmark by",round(dann_result["f1"]-coral_result["f1"],4))
else:
    print("NO F1 IMPROVEMENT: DANN did not beat the exact-split CORAL + balanced benchmark.")
    print("Do NOT tune DANN against CICIDS labels. Treat this as a failed intervention and move to the next principled adaptation mechanism.")


### Interpretation guide

- **DANN F1 > CORAL + balanced:** this is a real cross-dataset improvement under the same frozen target split. Report F1, Macro-F1, PR-AUC, ROC-AUC, precision, recall and FPR together.
- **DANN PR-AUC/ROC-AUC improves but F1 does not:** the representation is improving ranking but the decision boundary is still miscalibrated; do not claim overall success.
- **DANN does not improve ranking or F1:** stop adding random feature sets. The next scientifically justified direction would be **class-conditional/domain-specific adaptation**, because the earlier attack-family analysis showed that global alignment does not treat all attack families equally.

The key difference from the earlier phases is that this notebook tests a new mechanism—**learning the representation itself to be domain-invariant**—under a single fixed, apples-to-apples evaluation protocol.